# $\color{#dda}{\text{}}$ 

## $\color{#dda}{\text{Demo}}$

- **Modelo:** Ridge Regression (cargado desde disco)
- **Entrada:** 4 medidas antropométricas tomadas a 4 personas nuevas
- **Salida:** estatura predicha (cm)

In [2]:
import joblib
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn

RUTA_MODELO = Path("modelo_ridge_estatura.joblib")
RUTA_META   = Path("modelo_ridge_estatura_meta.joblib")

for ruta in (RUTA_MODELO, RUTA_META):
    if not ruta.exists():
        raise FileNotFoundError(
            f"Falta {ruta.resolve()}.\n"
            f"Ejecuta train_estatura.ipynb y asegúrate de que los .joblib "
            f"estén en el mismo directorio que este notebook."
        )

modelo = joblib.load(RUTA_MODELO)
meta   = joblib.load(RUTA_META)

# Chequeo de compatibilidad de versión (solo advertencia, no rompe)
if meta.get("sklearn_version") != sklearn.__version__:
    print(f"⚠ Modelo entrenado con sklearn {meta['sklearn_version']}, "
          f"ambiente actual {sklearn.__version__}. Verifica predicciones.")

print(f"✔ Modelo cargado desde: {RUTA_MODELO.resolve()}")
print(f"  Entrenado el        : {meta['fecha_entrenamiento']}")
print(f"  n train / test      : {meta['n_train']} / {meta['n_test']}")
print(f"  α óptimo            : {meta['alpha_optimo']:.4f}")
print(f"  MAE test (esperado) : {meta['mae_test']:.2f} cm")
print(f"  MAE baseline        : {meta['mae_baseline']:.2f} cm")
print(f"  Columnas esperadas  : {meta['columnas_X']}")

✔ Modelo cargado desde: C:\Users\hecto\OneDrive\Escritorio\ITESO\6to Semestre\EO\Exámenes\modelo_ridge_estatura.joblib
  Entrenado el        : 2026-10-19T12:18:16
  n train / test      : 38 / 10
  α óptimo            : 0.6368
  MAE test (esperado) : 1.39 cm
  MAE baseline        : 6.04 cm
  Columnas esperadas  : ['envergadura_cm', 'longitud_antebrazo_cm', 'longitud_mano_cm', 'talla_zapato_mx']


## $\color{#dda}{\text{Demostración en vivo (4 personas nuevas)}}$ 
#### $\color{#dda}{\text{Hola mundo}}$ 

| Variable | Cómo se mide |
|---|---|
| `envergadura_cm` | De la punta del dedo medio izquierdo a la del derecho, brazos extendidos en cruz, contra la pared |
| `longitud_antebrazo_cm` | Del codo (olecranon) a la apófisis estiloides de la muñeca, brazo flexionado 90° |
| `longitud_mano_cm` | De la base de la muñeca a la punta del dedo medio, mano extendida |
| `talla_zapato_mx` | Talla del calzado del sujeto (sistema MX) |

In [3]:
COLUMNAS_X = meta["columnas_X"]

def predecir_estatura(envergadura_cm, longitud_antebrazo_cm,
                      longitud_mano_cm, talla_zapato_mx):
    """Predice estatura (cm) a partir de las 4 medidas. Valida orden de columnas."""
    entrada = pd.DataFrame([{
        "envergadura_cm": envergadura_cm,
        "longitud_antebrazo_cm": longitud_antebrazo_cm,
        "longitud_mano_cm": longitud_mano_cm,
        "talla_zapato_mx": talla_zapato_mx,
    }])[COLUMNAS_X]                     # fuerza el orden de entrenamiento
    return float(modelo.predict(entrada)[0])


def reporte_persona(nombre, envergadura_cm, longitud_antebrazo_cm,
                    longitud_mano_cm, talla_zapato_mx):
    p = predecir_estatura(envergadura_cm, longitud_antebrazo_cm,
                          longitud_mano_cm, talla_zapato_mx)
    print(f"{nombre:12s} → {p:6.1f} cm  (±{meta['mae_test']:.1f} cm esperado)")
    return p

# $\color{#dda}{\text{SHOWTIME}}$

In [4]:
nuevas_personas = pd.DataFrame([
    {"nombre": "p1", "envergadura_cm": 172.0, "longitud_antebrazo_cm": 26.0,
     "longitud_mano_cm": 18.0, "talla_zapato_mx": 26.0},
    {"nombre": "p2", "envergadura_cm": 160.0, "longitud_antebrazo_cm": 24.0,
     "longitud_mano_cm": 17.0, "talla_zapato_mx": 23.5},
    {"nombre": "p3", "envergadura_cm": 181.0, "longitud_antebrazo_cm": 28.0,
     "longitud_mano_cm": 19.0, "talla_zapato_mx": 27.5},
    {"nombre": "p4", "envergadura_cm": 166.0, "longitud_antebrazo_cm": 25.0,
     "longitud_mano_cm": 17.5, "talla_zapato_mx": 24.5},
])

In [5]:
nuevas_personas["estatura_predicha_cm"] = [
    predecir_estatura(r.envergadura_cm, r.longitud_antebrazo_cm,
                      r.longitud_mano_cm, r.talla_zapato_mx)
    for r in nuevas_personas.itertuples()
]
nuevas_personas["estatura_predicha_cm"] = nuevas_personas["estatura_predicha_cm"].round(1)

print("Predicciones de la demo:\n")
for r in nuevas_personas.itertuples():
    reporte_persona(r.nombre, r.envergadura_cm, r.longitud_antebrazo_cm,
                    r.longitud_mano_cm, r.talla_zapato_mx)

display(nuevas_personas)

Predicciones de la demo:

p1           →  171.2 cm  (±1.4 cm esperado)
p2           →  160.3 cm  (±1.4 cm esperado)
p3           →  179.2 cm  (±1.4 cm esperado)
p4           →  165.4 cm  (±1.4 cm esperado)


,nombre,envergadura_cm,longitud_antebrazo_cm,longitud_mano_cm,talla_zapato_mx,estatura_predicha_cm
0,p1,172.0,26.0,18.0,26.0,171.2
1,p2,160.0,24.0,17.0,23.5,160.3
2,p3,181.0,28.0,19.0,27.5,179.2
3,p4,166.0,25.0,17.5,24.5,165.4


In [6]:
# e = float(input("Envergadura (cm): "))
# a = float(input("Longitud antebrazo (cm): "))
# m = float(input("Longitud mano (cm): "))
# z = float(input("Talla zapato MX: "))
# print(f"\n>>> Estatura predicha: {predecir_estatura(e, a, m, z):.1f} cm")

In [7]:
# Estaturas reales
nuevas_personas["estatura_real_cm"] = [None, None, None, None]

if nuevas_personas["estatura_real_cm"].notna().all():
    from sklearn.metrics import mean_absolute_error
    mae_demo = mean_absolute_error(
        nuevas_personas["estatura_real_cm"],
        nuevas_personas["estatura_predicha_cm"]
    )
    print(f"\nMAE de la demo (n=4): {mae_demo:.2f} cm")
    print(f"Puntos obtenidos (1/MAE): {1/mae_demo:.3f}")
else:
    print("Pon bien la data wei")

Pon bien la data wei


In [9]:
# import joblib
# joblib.dump(modelo, "modelo_ridge_estatura.joblib")
# print("Modelo guardado en modelo_ridge_estatura.joblib")

### $\color{#dda}{\text{Conclusiones}}$

1. **EDA:** `envergadura_cm` es el predictor dominante (r≈0.95). Los cuatro
   predictores presentan multicolinealidad, lo que motiva regularización L2.
2. **Modelo:** Ridge con estandarización y α elegido por validación cruzada de 5 folds.
3. **Desempeño:** MAE en prueba ≈ 2-3 cm, frente a ≈ 7-8 cm del baseline
   (predecir siempre la media del train) → reducción del error de ~65-70 %.
   En la gráfica real vs predicho, los puntos se pegan a la línea y=x.
4. **Limitaciones:**
   - n = 48 es pequeño; el intervalo de confianza del MAE es amplio.
   - Se excluyó `sexo` por restricción del proyecto; incluirlo reduciría el MAE
     pero rompería el supuesto de "solo medidas corporales".
   - La predicción es para población adulta joven (18-37 años) similar a la muestra.
5. **Demo en vivo:** se espera un error de ±3 cm por persona, lo que da
   aproximadamente 0.33-0.5 puntos por caso en la escala 1/MAE.